In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
transform = transforms.ToTensor()

train_data = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_data = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

train_loader = DataLoader(
    train_data,
    batch_size=64,
    shuffle=True
)

test_loader = DataLoader(
    test_data,
    batch_size=64,
    shuffle=False
)

100%|██████████| 9.91M/9.91M [01:52<00:00, 87.8kB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 115kB/s]
100%|██████████| 1.65M/1.65M [00:04<00:00, 410kB/s]
100%|██████████| 4.54k/4.54k [00:00<?, ?B/s]


In [4]:
class ResidualBlock(nn.Module):

    def __init__(self, channels):
        super().__init__()

        self.conv1 = nn.Conv2d(
            channels,
            channels,
            kernel_size=3,
            padding=1
        )

        self.relu = nn.ReLU()

        self.conv2 = nn.Conv2d(
            channels,
            channels,
            kernel_size=3,
            padding=1
        )

    def forward(self, x):

        identity = x

        out = self.conv1(x)
        out = self.relu(out)

        out = self.conv2(out)

        out = out + identity

        out = self.relu(out)

        return out

In [5]:
class SimpleResNet(nn.Module):

    def __init__(self):
        super().__init__()

        self.conv = nn.Conv2d(
            1,
            32,
            kernel_size=3,
            padding=1
        )

        self.relu = nn.ReLU()

        self.block1 = ResidualBlock(32)
        self.block2 = ResidualBlock(32)

        self.pool = nn.AdaptiveAvgPool2d((1, 1))

        self.fc = nn.Linear(32, 10)

    def forward(self, x):

        x = self.conv(x)
        x = self.relu(x)

        x = self.block1(x)
        x = self.block2(x)

        x = self.pool(x)

        x = torch.flatten(x, 1)

        x = self.fc(x)

        return x


In [6]:
model = SimpleResNet().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

In [7]:
epochs = 5

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"Loss: {total_loss / len(train_loader):.4f}"
    )

Epoch [1/5] Loss: 1.2142
Epoch [2/5] Loss: 0.3827
Epoch [3/5] Loss: 0.2548
Epoch [4/5] Loss: 0.1988
Epoch [5/5] Loss: 0.1660


In [8]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()


accuracy = 100 * correct / total

print(f"Test Accuracy: {accuracy:.2f}%")

Test Accuracy: 96.61%


In [ ]:
class SingleClassResNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv_entry = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.relu = nn.ReLU()


        self.b1_conv1 = nn.Conv2d(32, 32, kernel_size=3, padding=1)
        self.b1_conv2 = nn.Conv2d(32, 32, kernel_size=3, padding=1)

        self.b2_conv1 = nn.Conv2d(32, 32, kernel_size=3, padding=1)
        self.b2_conv2 = nn.Conv2d(32, 32, kernel_size=3, padding=1)

        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(32, 10)

    def forward(self, x):
        x = self.relu(self.conv_entry(x))

        identity1 = x
        out1 = self.relu(self.b1_conv1(x))
        out1 = self.b1_conv2(out1)
        x = self.relu(out1 + identity1)


        identity2 = x
        out2 = self.relu(self.b2_conv1(x))
        out2 = self.b2_conv2(out2)
        x = self.relu(out2 + identity2)

        x = self.pool(x)
        x = torch.flatten(x, 1)
        return self.fc(x)